In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from matplotlib import pyplot as plt
from silx.io.dictdump import h5todict

In [ ]:
import sys
sys.path.append('/home/zanardi/Codes/TC/HyperNet/hypernet')
sys.path.append('/home/zanardi/Codes/ML/PyCOMET/pycomet')

from hypernet.src.general import utils as hyp_utils
thermophysics = hyp_utils.get_thermophysics_mod()
mix_mod = thermophysics.reaction_thermo.mixture
sp_th_mod = thermophysics.specie_thermo.specie_thermo

In [ ]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET/chemreac/mt/O3/run.07/'
# Model
model_path = prefix + '/saving/mt_O3_mmopnn_siso/SavedModel/'
# Data
data_path = prefix + '/datagen/runs/data/test/'
box_header = ['t', 'X_O', 'X_O2', 'Th', 'Tv', 'rho', 'p', 'n', 'E']
case_num = 10

In [ ]:
specie_thermo = {
    0: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O2g.csv'
            }
        },
        'thermo': {
            'model': 'RigidRotorHarmonicOscillator',
            'Tr_index': 0,
            'Tv_index': 1
        }
    },
    1: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'Og.csv'
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    }
}

process = ['adiabatic', 'isochoric']

In [ ]:
def extract_physical_var(df):
    # Extract physical variables
    X = df[['X_O2', 'X_O']].to_numpy()
    T = df[['Th', 'Tv']].to_numpy()
    rho = df[['rho']].to_numpy()[0]
    t = df[['t']].to_numpy()
    # Update thermodynamics
    for s in sp_th.values():
        s.update(T)
    # Update mixture
    mix.update(X, var='X')
    # Compute needed physical variables
    Y = np.hstack([sp_th[k].specie.Y.numpy() for k in ['O2','O']])
    return tf.concat([Y*rho, T, t, Y, T], 1).numpy()

def encode(x):
    # Extracting inputs quantities to encode
    x = {k: tf.gather(x, [i], axis=1) for i, k in enumerate(['rho_m','rho_a','Th','Tv'])}
    # Allocate encoded quantities
    y = {}
    # Encode mass quantities
    y['rho'] = x['rho_m'] + x['rho_a']
    y['Y_m'] = x['rho_m'] / y['rho']
    # Encode energy quantities
    Y = tf.concat([x['rho_m'], x['rho_a']], 1) / y['rho']
    T = tf.concat([x['Th'], x['Tv']], 1)
    # Update thermodynamics
    for s in sp_th.values():
        s.update(T)
        mix.update(Y, var='Y')
    # Total energy and roto-translational temperature (heavy particles)
    y['e'] = mix._he()
    y['Th'] = x['Th']
    # Compose and return outputs
    return tf.concat([y['Y_m'], y['rho'], y['Th'], y['e']], 1).numpy()

In [ ]:
# Specie thermodynamics
sp_th = {}
for index, s in specie_thermo.items():
    args = {**s, **dict(index=index, process=process)}
    sp_th_ = sp_th_mod.SpecieThermo(**args)
    sp_th[sp_th_.specie.name] = sp_th_
for name, s in sp_th.items():
    if s.specie.n_at > 1:
        molecule = name
    else:
        atom = name
# Mixture
mix = mix_mod.MultiComponent(sp_th)

In [ ]:
# Load test case
case = h5todict(data_path + '/cases.hdf5')[str(int(case_num)).zfill(6)]
case = pd.DataFrame(case, columns=box_header)
ic = pd.read_csv(data_path + '/ics.csv').to_numpy()[case_num-1]

In [ ]:
# Test inputs encoding
X = extract_physical_var(case)
y = encode(X)
print('Are they close?', all(np.isclose(y[0], ic, rtol=1e-06, atol=1e-08)))

In [ ]:
# Load model
model = tf.keras.models.load_model(filepath=model_path)

In [ ]:
# Compose inputs and true values
cols = {
    'x': [0,1,2,3,4],
    'y': [5,6,7,8]
}
ic = np.tile(np.take(X, cols['x'][:-1], axis=1)[0], (len(X), 1))
t = np.take(X, [cols['x'][-1]], axis=1)
x = np.hstack([ic,t])
y_true = np.take(X, cols['y'], axis=1)

In [ ]:
y_pred = model(x).numpy()

In [ ]:
# Plot mass
plt.semilogx(t, np.take(y_true, [0,1], axis=1))
plt.semilogx(t, np.take(y_pred, [0,1], axis=1), '--')

In [ ]:
# Plot temperature
plt.semilogx(t, np.take(y_true, [2,3], axis=1))
plt.semilogx(t, np.take(y_pred, [2,3], axis=1), '--')